In [1]:
!pip install -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [2]:
import torch
import tensorflow as tf

print("PyTorch:", torch.__version__)
print("TensorFlow:", tf.__version__)

PyTorch: 2.11.0+cpu
TensorFlow: 2.20.0


In [3]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="openai-community/gpt2"
)

prompt = "Artificial Intelligence is"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1
)

print(result[0]["generated_text"])

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Artificial Intelligence is an industry that relies on artificial intelligence to improve its own productivity and reduce the cost of work.

The industry is also in a good position to meet demand from consumers and businesses whose productivity relies on artificial intelligence.

"We are in a


In [5]:
text = """
Artificial intelligence is a branch of computer science.
Machine learning allows computers to learn from data.
Deep learning uses neural networks to solve complex problems.
Generative AI can create new text, images, audio and code.
Natural language processing allows computers to understand human language.
Large language models can generate human-like text.
Transformers are widely used in modern language models.
Artificial intelligence is transforming healthcare and education.
Machine learning is used for prediction and classification.
Deep learning is useful for computer vision and natural language processing.
Generative AI is becoming an important technology in software development.
Large language models learn patterns from large amounts of text.
Text generation models predict the next token in a sequence.
Hugging Face provides tools for building and using machine learning models.
Python is one of the most popular languages for artificial intelligence.
"""

In [6]:
with open("ai_dataset.txt", "w") as f:
    f.write(text)

In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "text",
    data_files={"train": "ai_dataset.txt"}
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 16
    })
})


In [8]:
print(dataset["train"][0])

{'text': ''}


In [9]:
from transformers import AutoTokenizer

model_name = "openai-community/gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [10]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

In [11]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 16
    })
})


In [12]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)

model.config.pad_token_id = tokenizer.pad_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [13]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./text_generation_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

In [15]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

In [16]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.297901
20,1.940270


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24, training_loss=2.4641529321670532, metrics={'train_runtime': 113.6637, 'train_samples_per_second': 0.422, 'train_steps_per_second': 0.211, 'total_flos': 281705472000.0, 'train_loss': 2.4641529321670532, 'epoch': 3.0})

In [17]:
model.save_pretrained("./my_text_generator")
tokenizer.save_pretrained("./my_text_generator")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_text_generator/tokenizer_config.json',
 './my_text_generator/tokenizer.json')

In [18]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="./my_text_generator",
    tokenizer="./my_text_generator"
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [19]:
prompt = "Artificial intelligence"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'num_return_sequences', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Artificial intelligence has rapidly arrived in computers, making it possible to solve complex problems. But human-like machine learning is particularly powerful. Machines learn from data. What's more, machines learn from natural language processing. This has led to a number of new computer-


In [20]:
max_new_tokens=50

In [21]:
temperature=0.7

In [22]:
temperature=1.0

In [23]:
do_sample=True

In [24]:
num_return_sequences=3

In [25]:
result = generator(
    "Machine learning",
    max_new_tokens=50,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.8
)

for output in result:
    print(output["generated_text"])
    print()

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Machine learning is used to understand complex visual and auditory information. Learning is used to create complex data from text, images, music and other forms of information.

To understand human language, human-like systems are increasingly used in artificial intelligence. Humans can learn

Machine learning is used to solve mathematical problems. In this paper, we focus on neural networks and machine learning. The basic idea is that computers solve problems using techniques similar to human-like models. Some of these neural networks are trained on complex data. For example

Machine learning can be used to solve complex problems. This is also a fundamental field of computer science.

Machine Learning is one of the most important tools for artificial intelligence. It is used to solve complex problems. This is also a fundamental field of computer science



In [26]:
# ============================================
# TEXT GENERATION USING HUGGING FACE
# ============================================

# Install libraries
!pip install -U transformers datasets accelerate

# Import libraries
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    pipeline
)

# Model
model_name = "openai-community/gpt2"

# Create sample dataset
text = """
Artificial intelligence is a branch of computer science.
Machine learning allows computers to learn from data.
Deep learning uses neural networks to solve complex problems.
Generative AI can create new text, images, audio and code.
Natural language processing allows computers to understand human language.
Large language models can generate human-like text.
Transformers are widely used in modern language models.
Artificial intelligence is transforming healthcare and education.
Machine learning is used for prediction and classification.
Deep learning is useful for computer vision and natural language processing.
Generative AI is becoming an important technology.
Large language models learn patterns from large amounts of text.
Text generation models predict the next token in a sequence.
Hugging Face provides tools for building machine learning models.
Python is widely used for artificial intelligence.
"""

with open("ai_dataset.txt", "w") as f:
    f.write(text)

# Load dataset
dataset = load_dataset(
    "text",
    data_files={"train": "ai_dataset.txt"}
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# Tokenization
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Load pretrained GPT-2
model = AutoModelForCausalLM.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./text_generation_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

# Train
trainer.train()

# Save model
model.save_pretrained("./my_text_generator")
tokenizer.save_pretrained("./my_text_generator")

# Text generation
generator = pipeline(
    "text-generation",
    model="./my_text_generator",
    tokenizer="./my_text_generator"
)

# Generate
prompt = "Artificial intelligence"

result = generator(
    prompt,
    max_new_tokens=50,
    num_return_sequences=1,
    do_sample=True,
    temperature=0.8
)

print("\nGenerated Text:")
print(result[0]["generated_text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,3.382056
20,2.035420


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Generated Text:
Artificial intelligence is transforming industries. One of the most important innovations is machine learning. Machine learning has enabled new industries to create new products. Today, more computers are used for many tasks. These computers learn from human memory and data. They are able to solve complex
